# NB02 · Semana 4 — Extracción de URLhaus hacia la capa bronze

**Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07** · Base de la **Entrega 1 (E1)**

## Objetivo

Traer las URLs que distribuyen malware publicadas por **URLhaus (abuse.ch)** y guardarlas en la capa **bronze** del lake (Delta Lake en MinIO) cumpliendo tres condiciones:

| Requisito | Cómo se cumple acá |
|---|---|
| **Carga incremental** | El *watermark* es el `id` más alto que ya está en bronze. Solo se escriben URLs con `id` mayor. Si la API no alcanza a cubrir todo lo nuevo, se completa con el export de 30 días. |
| **Idempotencia** | No se trae lo que ya está: la segunda corrida seguida encuentra 0 URLs nuevas y no escribe nada. Siempre `append`, nunca `overwrite`. |
| **Manejo de errores** | Se separan fallas **transitorias** (timeout, caída de red, HTTP 5xx o 429: se reintenta 3 veces con espera creciente) de **permanentes** (clave inválida, endpoint retirado, respuesta con otro formato: se corta sin reintentar y sin escribir). Toda falla se avisa y queda registrada. |

La Auth-Key de URLhaus se lee de la variable de entorno `URLHAUS_AUTH_KEY`: no está escrita en el notebook.

In [1]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import BRONZE, STORAGE_OPTIONS, info
info()

# La Auth-Key vive en dags/.env (la escribe NB05). Si no está en el entorno del kernel, se toma de ahí.
if not os.environ.get("URLHAUS_AUTH_KEY"):
    _env = os.path.join(_d, "dags", ".env")
    if os.path.exists(_env):
        for _l in open(_env, encoding="utf-8"):
            if _l.startswith("URLHAUS_AUTH_KEY="):
                os.environ["URLHAUS_AUTH_KEY"] = _l.strip().split("=", 1)[1]
print("Auth-Key de URLhaus:", "cargada" if os.environ.get("URLHAUS_AUTH_KEY") else "FALTA")

Entorno detectado: servidor de la cátedra  (grupo: grupo07)
  Lake / bucket   : grupo07   →   bronze en s3://grupo07/bronze/series
  PostgreSQL      : base 'warehouse', usuario 'grupo07', host 'postgres'
  Material en     : /home/jovyan/work
Auth-Key de URLhaus: cargada


In [2]:
# Dependencias: el JupyterHub puede reiniciarse y perder lo instalado con pip
%pip install -q -r {os.path.join(_d, 'requirements.txt')}

Note: you may need to restart the kernel to use updated packages.


## 1. El *watermark*

Es el `id` más alto que ya está en bronze. URLhaus asigna ids crecientes: una URL nueva siempre tiene un id mayor que todas las anteriores. Se **deriva de los datos** (no se guarda en un archivo aparte), así que no puede desincronizarse de lo que realmente hay en bronze.

Si la tabla no existe, devuelve `None`: es la primera corrida. Cualquier otro error (MinIO caído, credenciales mal) **no** se toma como tabla vacía: se propaga, porque tratarlo como primera corrida haría una carga histórica encima de datos existentes.

In [3]:
import pyarrow.compute as pc
from deltalake import DeltaTable, write_deltalake
from deltalake.exceptions import TableNotFoundError


def leer_watermark():
    """Máximo id ya ingerido en bronze, o None si la tabla todavía no existe."""
    try:
        dt = DeltaTable(BRONZE, storage_options=STORAGE_OPTIONS)
    except TableNotFoundError:
        return None
    maximo = pc.max(dt.to_pyarrow_table(columns=["id"])["id"]).as_py()
    return None if maximo is None else int(maximo)


def contar_bronze():
    try:
        return DeltaTable(BRONZE, storage_options=STORAGE_OPTIONS).to_pyarrow_table(columns=["id"]).num_rows
    except TableNotFoundError:
        return 0


print("watermark actual:", leer_watermark(), "· filas en bronze:", contar_bronze())

watermark actual: 4069733 · filas en bronze: 189756


## 2. Pedidos a URLhaus con manejo de errores

`pedir()` es el único punto que habla con URLhaus. Verifica el código HTTP **antes** de leer el cuerpo y actúa según el tipo de falla:

| Qué pasa | Tipo | Qué hace |
|---|---|---|
| Timeout o error de conexión | transitoria | reintenta (hasta 3 intentos, esperando 5 s y 10 s) |
| HTTP 429 (demasiados pedidos) o 5xx | transitoria | reintenta igual |
| Sigue fallando después de 3 intentos | transitoria agotada | corta con `FallaTransitoria` |
| HTTP 401/403 (clave inválida), 404/410 (endpoint retirado), otro 4xx | permanente | corta en el acto con `FallaPermanente`: reintentar no lo arregla |
| 200 pero el cuerpo no es JSON, `query_status` no es `ok`, o faltan columnas | permanente | corta sin escribir nada en bronze |
| Filas con `id` o fecha ilegibles | dato malo | se descartan esas filas y se informa cuántas |

Cuando la corrida corta, la sección 4 la avisa con un mensaje `🚨 INGESTA FALLIDA` que dice el tipo de falla y qué hacer, y la anota en un registro de corridas. Bronze queda como estaba.

Los mensajes nunca muestran la Auth-Key (va dentro de la URL del export, así lo pide abuse.ch).

In [ ]:
import csv, io, time, zipfile
import pandas as pd
import pyarrow as pa
import requests

API_RECIENTES = "https://urlhaus-api.abuse.ch/v1/urls/recent/"
EXPORT_RECIENTE = "https://urlhaus-api.abuse.ch/v2/files/exports/{clave}/recent.csv"   # últimos 30 días
EXPORT_COMPLETO = "https://urlhaus-api.abuse.ch/v2/files/exports/{clave}/full.csv.zip"  # base completa

COLUMNAS = ["id", "date_added", "url", "url_status", "last_online",
            "threat", "tags", "urlhaus_link", "reporter"]
# Mismo esquema que escribe extraer_bronze en el DAG: las dos ingestas conviven en la misma tabla
ESQUEMA = pa.schema([
    ("id", pa.int64()), ("date_added", pa.timestamp("us", tz="UTC")),
    ("url", pa.string()), ("url_status", pa.string()), ("last_online", pa.string()),
    ("threat", pa.string()), ("tags", pa.string()),
    ("urlhaus_link", pa.string()), ("reporter", pa.string()),
])


class FallaTransitoria(Exception):
    """URLhaus no respondió bien, pero puede andar en un rato."""


class FallaPermanente(Exception):
    """Reintentar no sirve: hay que cambiar algo (clave, endpoint, código)."""


def _clave():
    clave = os.environ.get("URLHAUS_AUTH_KEY")
    if not clave:
        raise FallaPermanente("Falta la variable de entorno URLHAUS_AUTH_KEY")
    return clave


def pedir(url, intentos=3, espera=5, timeout=30, stream=False):
    clave = _clave()
    visible = url.replace(clave, "<AUTH-KEY>")
    motivo = ""
    for intento in range(1, intentos + 1):
        try:
            res = requests.get(url, headers={"Auth-Key": clave}, timeout=timeout, stream=stream)
        except (requests.Timeout, requests.ConnectionError) as e:
            motivo = type(e).__name__          # el texto de la excepción incluye la URL: no se muestra
        else:
            if res.status_code == 200:
                return res
            if res.status_code == 429 or res.status_code >= 500:
                motivo = f"HTTP {res.status_code}"
            else:
                raise FallaPermanente(f"URLhaus respondió HTTP {res.status_code} en {visible}: "
                                      "clave inválida, endpoint retirado o pedido mal formado")
        if intento < intentos:
            print(f"  intento {intento}/{intentos} falló ({motivo}); reintento en {espera * intento} s")
            time.sleep(espera * intento)
    raise FallaTransitoria(f"URLhaus no respondió bien en {intentos} intentos ({motivo}) · {visible}")


def _tags(valor):
    """Tags como texto separado por comas ("Mozi,elf"), igual que el CSV de URLhaus.
    La API los devuelve como lista: se unen."""
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return None
    partes = valor if isinstance(valor, (list, tuple)) else str(valor).split(",")
    partes = [str(p).strip() for p in partes if str(p).strip()]
    return ",".join(partes) or None


def _texto(valor):
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return None
    valor = str(valor).strip()
    return None if valor in ("", "None") else valor


def normalizar(df):
    """Valida columnas, tipa con el esquema de bronze, descarta filas ilegibles y deduplica por id."""
    faltan = set(COLUMNAS) - set(df.columns)
    if faltan:
        raise FallaPermanente(f"La respuesta no trae las columnas {sorted(faltan)}: cambió el formato")
    df = df[COLUMNAS].copy()
    df["id"] = pd.to_numeric(df["id"], errors="coerce")
    fechas = df["date_added"].astype(str).str.replace(" UTC", "", regex=False)
    df["date_added"] = pd.to_datetime(fechas, format="%Y-%m-%d %H:%M:%S", utc=True, errors="coerce")
    malas = df["id"].isna() | df["date_added"].isna()
    if malas.any():
        print(f"  {int(malas.sum())} filas descartadas por id o fecha ilegibles")
    df = df[~malas].copy()
    df["id"] = df["id"].astype("int64")
    df["tags"] = df["tags"].map(_tags)
    for col in ["url", "url_status", "last_online", "threat", "urlhaus_link", "reporter"]:
        df[col] = df[col].map(_texto)
    df = df.drop_duplicates(subset=["id"])
    return pa.Table.from_pandas(df, schema=ESQUEMA, preserve_index=False)

## 3. Extracción

Tres fuentes de URLhaus, con las mismas columnas:

| Fuente | Qué trae | Cuándo se usa |
|---|---|---|
| API `/v1/urls/recent/` | URLs de los últimos 3 días, **como máximo 1000** | Corrida normal: es un pedido chico |
| Export `recent.csv` | Todas las URLs de los últimos 30 días | Cuando la API no alcanza (ver abajo) |
| Export `full.csv.zip` | La base completa | Primera corrida, o si pasaron más de 30 días sin correr |

**Límite de la fuente.** La API no acepta un parámetro "desde" (ni por fecha ni por id) y corta en 1000 URLs. Por eso el watermark no viaja en el pedido sino que filtra la respuesta (`id > watermark`).

Ese tope de 1000 es poco: en agosto y septiembre de 2026 se reportaban unas 500 URLs por día, y en octubre casi 4.000. Si desde la última corrida entraron 2000 URLs, la API devuelve solo las 1000 más recientes y las otras 1000 se perderían sin que nadie se entere.

**Cómo se detecta.** Si la respuesta de la API no llega hasta el watermark (ninguna de sus URLs tiene un `id` que ya estaba en bronze), entre lo que ya tenemos y lo que devolvió puede haber un hueco. En ese caso la corrida avisa y trae lo nuevo del export de 30 días, que no tiene tope. Si tampoco ese export llega al watermark (más de un mes sin correr), usa la base completa. Así nunca queda un hueco, y en el caso normal el pedido sigue siendo chico.

El DAG de Airflow (NB05) usa siempre los exports, porque procesa por intervalo de fechas y esta API no sirve para el pasado.

In [ ]:
def _leer_export(url):
    """Descarga un export CSV de URLhaus (recent.csv o full.csv.zip) y lo devuelve normalizado."""
    res = pedir(url, timeout=600, stream=True)
    datos = io.BytesIO()
    for bloque in res.iter_content(1 << 20):
        datos.write(bloque)
    if zipfile.is_zipfile(datos):
        zf = zipfile.ZipFile(datos)
        crudo = zf.open(zf.namelist()[0])
    else:
        datos.seek(0)
        crudo = datos
    # Las columnas se ubican por NOMBRE, con el encabezado del export ("# id,dateadded,url,...",
    # o una primera fila con los nombres): si URLhaus las cambia de orden o agrega una, se lee
    # igual; si falta alguna, normalizar() corta con FallaPermanente.
    alias = {"dateadded": "date_added", "urlhaus_reference": "urlhaus_link"}
    encabezado = []

    def sin_comentarios(lineas):
        for l in lineas:
            if l.startswith("#"):
                campos = [c.strip().strip('"').lower() for c in l.lstrip("#").strip().split(",")]
                if "id" in campos and len(campos) >= 5:
                    encabezado[:] = [alias.get(c, c) for c in campos]
            else:
                yield l

    lineas = io.TextIOWrapper(crudo, encoding="utf-8", errors="replace", newline="")
    filas = list(csv.reader(sin_comentarios(lineas)))
    nombres = {c.strip().lower() for c in filas[0]} if filas else set()
    if "id" in nombres and nombres & {"dateadded", "date_added"}:      # encabezado como fila común
        encabezado[:] = [alias.get(c.strip().lower(), c.strip().lower()) for c in filas[0]]
        filas = filas[1:]
    if not encabezado:
        print("  el export no trae encabezado: se asume el orden estándar de URLhaus")
        encabezado[:] = COLUMNAS
    elif encabezado[:len(COLUMNAS)] != COLUMNAS:
        print(f"  columnas en otro orden o con agregados: se leen por nombre ({encabezado})")
    filas = [r[:len(encabezado)] for r in filas if len(r) >= len(encabezado)]
    if not filas:
        raise FallaPermanente("El export no trae filas con el formato esperado (¿página de error?)")
    return normalizar(pd.DataFrame(filas, columns=encabezado))


def _llega_al_watermark(tabla, desde):
    """True si la tabla incluye alguna URL que ya estaba en bronze: entonces no hay hueco."""
    return tabla.num_rows > 0 and pc.min(tabla["id"]).as_py() <= desde


def extraer_recientes(desde, api=API_RECIENTES):
    """URLs con id mayor al watermark. Devuelve (tabla, fuente usada)."""
    res = pedir(api)
    try:
        cuerpo = res.json()
    except ValueError:
        raise FallaPermanente("La API devolvió 200 pero el cuerpo no es JSON") from None
    estado = cuerpo.get("query_status")
    if estado not in ("ok", "no_results"):
        raise FallaPermanente(f"La API respondió query_status={estado!r}")
    urls = cuerpo.get("urls") or []
    if urls:
        df = pd.DataFrame(urls).rename(columns={"urlhaus_reference": "urlhaus_link"})
        if "last_online" not in df.columns:          # la API no lo trae; el CSV sí
            df["last_online"] = None
        tabla = normalizar(df)
    else:
        tabla = ESQUEMA.empty_table()
    print(f"  la API devolvió {tabla.num_rows} URLs (tope: 1000)")

    if _llega_al_watermark(tabla, desde):
        tabla = tabla.filter(pc.greater(tabla["id"], desde))
        print(f"  cubre todo desde el watermark; con id > {desde}: {tabla.num_rows}")
        return tabla, "api"

    # Hueco posible: más URLs nuevas de las que devuelve la API, o más de 3 días sin correr
    print(f"  ⚠️  La API no llega hasta el watermark (id {desde}): pueden faltar URLs. "
          "Completo con el export de los últimos 30 días.")
    tabla = _leer_export(EXPORT_RECIENTE.format(clave=_clave()))
    fuente = "recent.csv"
    if not _llega_al_watermark(tabla, desde):
        print("  ⚠️  Tampoco el export de 30 días llega al watermark: uso la base completa.")
        tabla, fuente = _leer_export(EXPORT_COMPLETO.format(clave=_clave())), "full.csv.zip"
    tabla = tabla.filter(pc.greater(tabla["id"], desde))
    print(f"  {fuente}: con id > {desde}: {tabla.num_rows}")
    return tabla, fuente


def extraer_historico():
    """Base completa de URLhaus (solo para la primera corrida)."""
    tabla = _leer_export(EXPORT_COMPLETO.format(clave=_clave()))
    print(f"  histórico: {tabla.num_rows} URLs")
    return tabla, "full.csv.zip"

## 4. Escribir en bronze, avisar y registrar

`ingerir()` decide qué hacer según el watermark. Siempre escribe en modo `append` y solo filas nuevas: no pisa nada de lo que ya estaba.

`correr_ingesta()` la envuelve y deja constancia de cada corrida, salga bien o mal, en una tabla Delta aparte (`control/ingestas`, fuera de bronze para no mezclarla con los datos). Si la corrida falla:

* muestra un aviso `🚨 INGESTA FALLIDA` con el tipo de falla y qué hacer;
* anota la falla en el registro, con el watermark del momento;
* vuelve a lanzar el error, así la celda queda en rojo y quien la llame (un script o un orquestador) se entera de que falló.

In [ ]:
from datetime import datetime, timezone

CONTROL = BRONZE.split("/bronze/")[0] + "/control/ingestas"
ESQUEMA_CONTROL = pa.schema([
    ("inicio", pa.timestamp("us", tz="UTC")), ("fin", pa.timestamp("us", tz="UTC")),
    ("resultado", pa.string()), ("fuente", pa.string()), ("filas_nuevas", pa.int64()),
    ("watermark_antes", pa.int64()), ("watermark_despues", pa.int64()), ("detalle", pa.string()),
])
QUE_HACER = {
    "falla_transitoria": "URLhaus o la red no respondieron. Volver a correr en un rato.",
    "falla_permanente": "Reintentar no lo arregla: revisar la Auth-Key en dags/.env, el endpoint "
                        "o si cambió el formato de URLhaus.",
}


def ingerir(api=API_RECIENTES):
    """Devuelve (filas escritas, fuente usada)."""
    wm = leer_watermark()
    if wm is None:
        print("Bronze vacío: primera corrida, carga histórica completa")
        nuevos, fuente = extraer_historico()
    else:
        print(f"Watermark: id {wm}")
        nuevos, fuente = extraer_recientes(wm, api)
    if nuevos.num_rows == 0:
        print("Sin URLs nuevas: no se escribe nada en bronze")
        return 0, fuente
    try:   # se adapta al esquema real de bronze (lo comparte con el DAG)
        destino = DeltaTable(BRONZE, storage_options=STORAGE_OPTIONS).to_pyarrow_dataset().schema
        nuevos = nuevos.select(destino.names).cast(destino)
    except TableNotFoundError:
        pass
    write_deltalake(BRONZE, nuevos, mode="append", schema_mode="merge", storage_options=STORAGE_OPTIONS)
    ids = nuevos["id"]
    print(f"{nuevos.num_rows} URLs nuevas en bronze (ids {pc.min(ids).as_py()} a {pc.max(ids).as_py()})")
    return nuevos.num_rows, fuente


def registrar(inicio, resultado, fuente, filas, wm_antes, detalle=""):
    fila = pa.Table.from_pylist([{
        "inicio": inicio, "fin": datetime.now(timezone.utc), "resultado": resultado, "fuente": fuente,
        "filas_nuevas": filas, "watermark_antes": wm_antes, "watermark_despues": leer_watermark(),
        "detalle": detalle[:500]}], schema=ESQUEMA_CONTROL)
    try:
        write_deltalake(CONTROL, fila, mode="append", storage_options=STORAGE_OPTIONS)
    except Exception as e:   # si no se puede anotar, se avisa pero no se tapa el resultado de la corrida
        print(f"  (no se pudo escribir el registro de corridas: {type(e).__name__})")


def correr_ingesta(api=API_RECIENTES):
    inicio, wm_antes = datetime.now(timezone.utc), leer_watermark()
    try:
        filas, fuente = ingerir(api)
    except (FallaTransitoria, FallaPermanente) as e:
        tipo = "falla_transitoria" if isinstance(e, FallaTransitoria) else "falla_permanente"
        print(f"\n🚨 INGESTA FALLIDA ({tipo}): {e}\n   Bronze no se modificó. {QUE_HACER[tipo]}")
        registrar(inicio, tipo, None, 0, wm_antes, str(e))
        raise
    registrar(inicio, "ok", fuente, filas, wm_antes)
    return filas


def ver_registro(n=10):
    try:
        df = DeltaTable(CONTROL, storage_options=STORAGE_OPTIONS).to_pandas()
    except TableNotFoundError:
        return pd.DataFrame(columns=ESQUEMA_CONTROL.names)
    return df.sort_values("inicio", ascending=False).head(n).reset_index(drop=True)

## 5. Demostración: incremental e idempotente

Dos corridas seguidas. La primera trae lo nuevo; la segunda arranca de un watermark ya actualizado y no encuentra nada. No es que deduplique: **no pide escribir lo que ya tiene**.

In [ ]:
antes = contar_bronze()
print("--- Corrida 1 ---")
n1 = correr_ingesta()
print("\n--- Corrida 2 (inmediatamente después) ---")
n2 = correr_ingesta()
despues = contar_bronze()

print(f"\nFilas en bronze: {antes} → {despues} (corrida 1: +{n1}, corrida 2: +{n2})")
assert n2 == 0, "La segunda corrida no debería traer nada"
assert despues == antes + n1, "Bronze creció distinto de lo que se escribió"
print("✅ Incremental e idempotente: la segunda corrida no agregó filas")

## 6. Demostración: una falla permanente se avisa, no se reintenta y queda registrada

Se corre la ingesta contra un endpoint que no existe. URLhaus devuelve 404: `pedir()` corta en el primer intento, no se escribe nada en bronze, aparece el aviso `🚨 INGESTA FALLIDA` y la falla queda en el registro. (Una falla transitoria se ve como `intento 1/3 falló (...); reintento en 5 s`, y si se agotan los 3 intentos termina en el mismo aviso con `falla_transitoria`.)

In [ ]:
antes = contar_bronze()
try:
    correr_ingesta(api="https://urlhaus-api.abuse.ch/v1/este-endpoint-no-existe/")
except FallaPermanente:
    pass   # ya se avisó y se registró; acá se atrapa solo para que el notebook siga
assert contar_bronze() == antes, "Una falla no debería tocar bronze"
print("\nFilas en bronze siguen siendo:", contar_bronze())

## 7. Registro de corridas

Cada corrida deja una fila: cuándo empezó y terminó, si salió bien o con qué falla, de qué fuente leyó, cuántas URLs nuevas escribió y el watermark antes y después. Es lo primero que se mira cuando algo no cuadra.

In [ ]:
ver_registro()